In [1]:
import fitz  # PyMuPDF
from tqdm.auto import tqdm
from pathlib import Path # Using pathlib for robust path handling


e:\Study\Capstone project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
document_path= [
    "Sanquo_Section1_Platform_Introduction.docx",
    "Sanquo_Section2_ConditionsOfUse.docx",
    "Sanquo_Section3_PaymentMethods.docx",
    "Sanquo_Section4_ReturnPolicy.docx",
    "Sanquo_Section5_PrivacyNote.docx",
    "Sanquo_Section6_ShippingAndDelivery.docx",
    "Sanquo_Section7_TechnicalAndDeviceIssues.docx",
    "Sanquo_Section8_ContactInformation.docx",
]

document_title1 = [
    "PlatformIntroduction",
    "ConditionsOfUse",
    "PaymentMethods",
    "ReturnPolicy",
    "PrivacyNote",
    "ShippingAndDelivery",
    "TechnicalAndDeviceIssues",
    "ContactInformation",
]

# Thử in ra kiểm tra nè
for file in document_path:
    print(file)

Sanquo_Section1_Platform_Introduction.docx
Sanquo_Section2_ConditionsOfUse.docx
Sanquo_Section3_PaymentMethods.docx
Sanquo_Section4_ReturnPolicy.docx
Sanquo_Section5_PrivacyNote.docx
Sanquo_Section6_ShippingAndDelivery.docx
Sanquo_Section7_TechnicalAndDeviceIssues.docx
Sanquo_Section8_ContactInformation.docx


In [3]:
def text_formatter(text: str) -> str:
    """Perform minor formatting on text."""
    # .replace('\n', ' ') is good. Chaining .strip() is also good.
    cleaned_text = text.replace("\n", " ").strip()
    
    # Add more cleaning here as needed (e.g., removing multiple spaces)
    # import re
    # cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
    return cleaned_text

def open_and_read_pdf(path: str, base_dir: Path, document_title1:str) -> dict:
    """
    Opens a single PDF, extracts text and metadata from each page.
    
    Args:
        document_title: The name of the document (e.g., "Introduction").
        base_dir: The parent directory containing the PDFs.
        
    Returns:
        A dictionary with the document name and a list of page data.
    """
    
    # 1. FIX: Use robust path handling (Pathlib or forward slashes)
    pdf_path = base_dir / f"{path}.pdf"
    
    # Check if the file actually exists before trying to open it
    if not pdf_path.exists():
        print(f"Warning: File not found, skipping: {pdf_path}")
        return {'name': document_title1, 'doc': [], 'error': 'File not found'}

    doc = fitz.open(pdf_path)
    pages_and_texts = []

    # 2. FIX: Added internal tqdm, but the outer one is more important
    for page_number, page in enumerate(doc):
        text = page.get_text()
        text = text_formatter(text=text)
        
        # 3. FIX: Use text.split() for accurate word count
        word_count = len(text.split())
        
        # 4. FIX: Removed the flawed sentence count. 
        # Add it back ONLY if you use a real tokenizer (e.g., nltk.sent_tokenize)
        
        pages_and_texts.append({
            "page_number": page_number,
            "page_char_count": len(text),
            "page_word_count": word_count, 
            "page_token_count": len(text) / 4, # This heuristic is fine
            "text": text
        })
    
    doc.close() # Good practice to close the document
    return {'name': document_title1, 'doc': pages_and_texts}

In [4]:

DATA_DIR = Path("../data/documents")

document_data = [] # Renamed from 'document' to be less ambiguous

# 5. FIX: Added tqdm to the OUTER loop to track overall progress
print(f"Processing {len(document_path)} PDF documents from {DATA_DIR}...")
for path,title in tqdm(zip(document_path, document_title1), total=len(document_path), desc="Processing PDFs"):
    
    # 6. FIX: Use .append() for clarity and efficiency
    document_data.append(open_and_read_pdf(path, DATA_DIR, title))

print("Processing complete.")
print(f"Successfully processed {len(document_data)} documents.")


Processing 8 PDF documents from ..\data\documents...


Processing PDFs: 100%|██████████| 8/8 [00:00<00:00, 34.38it/s]

Processing complete.
Successfully processed 8 documents.


Check document (pre - chunking)

In [5]:
n = len(document_data[0]["doc"])

for i in range(0,n):
    document_data[0]["doc"][i]['text'] = document_data[0]["doc"][i]['text'].replace("\u200b", ":")
    document_data[0]["doc"][i]['text'] = document_data[0]["doc"][i]['text'].replace("|", "")

document_data[0] # Introduction 

{'name': 'PlatformIntroduction',
 'doc': [{'page_number': 0,
   'page_char_count': 1933,
   'page_word_count': 302,
   'page_token_count': 483.25,
   'text': 'SANQUO  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 1: Introduction  1.1  Welcome to Sanquo  Welcome to Sanquo — Vietnam\'s dedicated fashion destination, built for customers who  believe that style is personal, and shopping should be effortless.  We created Sanquo with a single purpose: to provide a seamless, secure, and transparent  shopping experience that empowers you to discover and acquire the fashion products you  love with total confidence and ease. Whether you are exploring the latest trends, seeking a  wardrobe staple, or discovering an emerging Vietnamese designer, Sanquo is the place  where your style journey begins.  Your trust is the foundation upon which everything we do is built. We are committed to  earning it every day — through honest communication, a carefully 

In [6]:
n = len(document_data[1]["doc"])

for i in range(0,n):
    document_data[1]["doc"][i]['text'] = document_data[1]["doc"][i]['text'].replace("\u200b", ":")
    document_data[1]["doc"][i]['text'] = document_data[1]["doc"][i]['text'].replace("|", "")

document_data[1] 

{'name': 'ConditionsOfUse',
 'doc': [{'page_number': 0,
   'page_char_count': 2135,
   'page_word_count': 328,
   'page_token_count': 533.75,
   'text': "Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 2: Conditions of Use  These Conditions of Use govern the behaviour and responsibilities of all individuals who  access or interact with the Sanquo platform. By using Sanquo, you agree to conduct yourself  in accordance with the standards set out in this Section. These conditions exist to ensure  that Sanquo remains a safe, fair, and trustworthy environment for every customer, seller, and  partner who is part of our community.  2.1  User Eligibility  Access to Sanquo and the ability to create a registered account are subject to the following  eligibility requirements, which exist to ensure that all users possess the legal capacity to  enter into binding agreements under Vietnamese civil law.  To register an account independently, you m

In [7]:
n = len(document_data[2]["doc"])

for i in range(0,n):
    document_data[2]["doc"][i]['text'] = document_data[2]["doc"][i]['text'].replace("\u200b", ":")
    document_data[2]["doc"][i]['text'] = document_data[2]["doc"][i]['text'].replace("|", "")

document_data[2] 

{'name': 'PaymentMethods',
 'doc': [{'page_number': 0,
   'page_char_count': 1839,
   'page_word_count': 262,
   'page_token_count': 459.75,
   'text': 'Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 3: Payment Methods  Sanquo is committed to making every transaction as simple, secure, and transparent as  possible. This Section sets out the payment options available to customers, the security  standards that protect every transaction, the currency framework governing purchases, the  step-by-step payment and confirmation process, and the dedicated support available should  any payment issue arise. Customers are encouraged to read this Section carefully before  completing a purchase.  3.1  Accepted Payment Methods  Sanquo accepts the following forms of payment. All methods listed below have been  selected to ensure broad accessibility, operational reliability, and alignment with the  preferences of Vietnamese consumers.  •: Digital W

In [8]:
n = len(document_data[3]["doc"])

for i in range(0,n):
    document_data[3]["doc"][i]['text'] = document_data[3]["doc"][i]['text'].replace("\u200b", ":")
    document_data[3]["doc"][i]['text'] = document_data[3]["doc"][i]['text'].replace("|", "")

document_data[3] 

{'name': 'ReturnPolicy',
 'doc': [{'page_number': 0,
   'page_char_count': 2240,
   'page_word_count': 338,
   'page_token_count': 560.0,
   'text': 'Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 4: Return Policy  Sanquo\'s Return Policy is designed to give customers the confidence to shop freely, knowing  that a clear, fair, and transparent resolution process exists if a purchase does not meet their  expectations. This Section sets out the conditions under which returns are accepted, the  timeframes within which requests must be initiated, the resolution options available, the  allocation of return shipping costs, and the inspection process that governs every return.  Customers are encouraged to read this Section carefully before initiating a return request.  4.1  Return Eligibility  Not all items are eligible for return, and not all conditions of return are equivalent. To be  accepted, a returned item must satisfy each of the fo

In [9]:
n = len(document_data[4]["doc"])

for i in range(0,n):
    document_data[4]["doc"][i]['text'] = document_data[4]["doc"][i]['text'].replace("\u200b", ":")
    document_data[4]["doc"][i]['text'] = document_data[4]["doc"][i]['text'].replace("|", "")

document_data[4] # Privacy note

{'name': 'PrivacyNote',
 'doc': [{'page_number': 0,
   'page_char_count': 1788,
   'page_word_count': 252,
   'page_token_count': 447.0,
   'text': "Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 5: Privacy Note  Sanquo is committed to the responsible, transparent, and lawful handling of every  customer's personal data. This Privacy Note explains what information Sanquo collects, why  it is collected, how it is used, with whom it may be shared, and the rights that every user  holds over their own data. This Section is drafted in compliance with applicable Vietnamese  data protection legislation, including the Law on Cybersecurity 2018 and Decree  13/2023/ND-CP on Personal Data Protection, and reflects internationally recognised privacy  principles including those embodied in the General Data Protection Regulation (GDPR).  Customers are encouraged to read this Section carefully and to direct any questions to  privacy@Sanquo.vn.  5.1

In [10]:
n = len(document_data[5]["doc"])

for i in range(0,n):
    document_data[5]["doc"][i]['text'] = document_data[5]["doc"][i]['text'].replace("\u200b", ":")
    document_data[5]["doc"][i]['text'] = document_data[5]["doc"][i]['text'].replace("|", "")

document_data[5] 

{'name': 'ShippingAndDelivery',
 'doc': [{'page_number': 0,
   'page_char_count': 2204,
   'page_word_count': 321,
   'page_token_count': 551.0,
   'text': "Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 6: Shipping and Delivery  Sanquo is committed to delivering every order reliably, transparently, and on time. This  Section describes the shipping options available to customers, the estimated delivery  timelines by geographic zone, the process for tracking a shipment, the restrictions that apply  to certain addresses and product categories, and the procedures in place should a package  arrive damaged or fail to arrive at all. Customers are encouraged to review this Section  before placing an order to ensure that the delivery options available meet their requirements.  6.1  Shipping Options  Sanquo offers three shipping tiers, designed to accommodate a range of delivery priorities  and budgets. All options are fulfilled through San

In [11]:
n = len(document_data[6]["doc"])

for i in range(0,n):
    document_data[6]["doc"][i]['text'] = document_data[6]["doc"][i]['text'].replace("\u200b", ":")
    document_data[6]["doc"][i]['text'] = document_data[6]["doc"][i]['text'].replace("|", "")
document_data[6] 

{'name': 'TechnicalAndDeviceIssues',
 'doc': [{'page_number': 0,
   'page_char_count': 1583,
   'page_word_count': 225,
   'page_token_count': 395.75,
   'text': "Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 7: Technical and Device Issues  Sanquo is committed to delivering a stable, responsive, and technically reliable platform  experience. This Section sets out the system requirements for optimal Platform  performance, the procedures governing scheduled and emergency maintenance, practical  troubleshooting guidance for the most common technical issues encountered by customers,  and the scope of Sanquo's liability in relation to device-side and connectivity-related  disruptions.  7.1  System Requirements  To ensure the best possible experience on the Sanquo Platform — whether accessed via  web browser or the Sanquo mobile application — customers' devices should meet the  following technical specifications. Using unsupported brows

In [12]:
n = len(document_data[7]["doc"])

for i in range(0,n):
    document_data[7]["doc"][i]['text'] = document_data[7]["doc"][i]['text'].replace("\u200b", ":")
    document_data[7]["doc"][i]['text'] = document_data[7]["doc"][i]['text'].replace("|", "")
    
document_data[7] 

{'name': 'ContactInformation',
 'doc': [{'page_number': 0,
   'page_char_count': 1654,
   'page_word_count': 229,
   'page_token_count': 413.5,
   'text': 'Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 8: Contact Information  Sanquo is here whenever customers need assistance. Whether the matter relates to an  order, a payment, an account, a return, or any other aspect of the Platform experience, our  support team is ready to help through a range of accessible channels. This Section sets out  how to reach us, when we are available, where we are located, and the response time  commitments we hold ourselves to.  8.1  Customer Support Channels  Sanquo provides customer support through the following official channels. Customers are  encouraged to select the channel best suited to the urgency and nature of their enquiry.  Channel  Contact Detail  Best Used For  Hotline  +84 1234 1423  Urgent matters requiring  real-time voice assistance

Chunking 

In [13]:
from spacy.lang.en import English

nlp = English()

# Add a sentecnizer pipeline, see
nlp.add_pipe("sentencizer")

def split_sentence(pages_and_texts):
    for item in tqdm(pages_and_texts):
        item["sentences"] = list(nlp(item["text"]).sents)
        #Make sure all sentences are strings (the default type is a spaCy datatype)
        item["sentences"] = [str(sentence) for sentence in item["sentences"]]
        # Count
        item["page_sentence_count_spacy"] = len(item["sentences"])

In [14]:
def split_list(input_list:list, slice_size: int = 10) -> list[list[str]]:
    return [input_list[i:i+ slice_size] for i in range(0, len(input_list),slice_size)]

test_list = list(range(25))
split_list(test_list)

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
 [20, 21, 22, 23, 24]]

In [15]:
# Loop through pages and texts and split sentences into chunks
def chunking(pages_and_texts,num_sentence_chunk_size):
    for item in tqdm(pages_and_texts):
        item["sentences_chunks"] = split_list(input_list = item["sentences"], slice_size = num_sentence_chunk_size)
        item["num_chunks"] = len(item["sentences_chunks"])

## Introduction

In [16]:
pages_and_texts = document_data[0]['doc']
split_sentence(pages_and_texts)
# chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 3/3 [00:00<00:00, 63.04it/s]


[{'page_number': 0,
  'page_char_count': 1933,
  'page_word_count': 302,
  'page_token_count': 483.25,
  'text': 'SANQUO  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 1: Introduction  1.1  Welcome to Sanquo  Welcome to Sanquo — Vietnam\'s dedicated fashion destination, built for customers who  believe that style is personal, and shopping should be effortless.  We created Sanquo with a single purpose: to provide a seamless, secure, and transparent  shopping experience that empowers you to discover and acquire the fashion products you  love with total confidence and ease. Whether you are exploring the latest trends, seeking a  wardrobe staple, or discovering an emerging Vietnamese designer, Sanquo is the place  where your style journey begins.  Your trust is the foundation upon which everything we do is built. We are committed to  earning it every day — through honest communication, a carefully curated selection of  products, and a platfor

## Condition of use


In [17]:
pages_and_texts = document_data[1]['doc']
split_sentence(pages_and_texts)
# chunking(pages_and_texts,13)
pages_and_texts

100%|██████████| 5/5 [00:00<00:00, 49.85it/s]


[{'page_number': 0,
  'page_char_count': 2135,
  'page_word_count': 328,
  'page_token_count': 533.75,
  'text': "Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 2: Conditions of Use  These Conditions of Use govern the behaviour and responsibilities of all individuals who  access or interact with the Sanquo platform. By using Sanquo, you agree to conduct yourself  in accordance with the standards set out in this Section. These conditions exist to ensure  that Sanquo remains a safe, fair, and trustworthy environment for every customer, seller, and  partner who is part of our community.  2.1  User Eligibility  Access to Sanquo and the ability to create a registered account are subject to the following  eligibility requirements, which exist to ensure that all users possess the legal capacity to  enter into binding agreements under Vietnamese civil law.  To register an account independently, you must be at least eighteen (18) years of a

## Payment


In [18]:
pages_and_texts = document_data[2]['doc']
split_sentence(pages_and_texts)
# chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 5/5 [00:00<00:00, 82.95it/s]


[{'page_number': 0,
  'page_char_count': 1839,
  'page_word_count': 262,
  'page_token_count': 459.75,
  'text': 'Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 3: Payment Methods  Sanquo is committed to making every transaction as simple, secure, and transparent as  possible. This Section sets out the payment options available to customers, the security  standards that protect every transaction, the currency framework governing purchases, the  step-by-step payment and confirmation process, and the dedicated support available should  any payment issue arise. Customers are encouraged to read this Section carefully before  completing a purchase.  3.1  Accepted Payment Methods  Sanquo accepts the following forms of payment. All methods listed below have been  selected to ensure broad accessibility, operational reliability, and alignment with the  preferences of Vietnamese consumers.  •: Digital Wallets: Sanquo supports payments via Za

## Return Policy

In [19]:
pages_and_texts = document_data[3]['doc']
split_sentence(pages_and_texts)
# chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 5/5 [00:00<00:00, 105.84it/s]


[{'page_number': 0,
  'page_char_count': 2240,
  'page_word_count': 338,
  'page_token_count': 560.0,
  'text': 'Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 4: Return Policy  Sanquo\'s Return Policy is designed to give customers the confidence to shop freely, knowing  that a clear, fair, and transparent resolution process exists if a purchase does not meet their  expectations. This Section sets out the conditions under which returns are accepted, the  timeframes within which requests must be initiated, the resolution options available, the  allocation of return shipping costs, and the inspection process that governs every return.  Customers are encouraged to read this Section carefully before initiating a return request.  4.1  Return Eligibility  Not all items are eligible for return, and not all conditions of return are equivalent. To be  accepted, a returned item must satisfy each of the following criteria without exception.  

## Privacy note

In [20]:
pages_and_texts = document_data[4]['doc']
split_sentence(pages_and_texts)
# chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 7/7 [00:00<00:00, 120.08it/s]


[{'page_number': 0,
  'page_char_count': 1788,
  'page_word_count': 252,
  'page_token_count': 447.0,
  'text': "Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 5: Privacy Note  Sanquo is committed to the responsible, transparent, and lawful handling of every  customer's personal data. This Privacy Note explains what information Sanquo collects, why  it is collected, how it is used, with whom it may be shared, and the rights that every user  holds over their own data. This Section is drafted in compliance with applicable Vietnamese  data protection legislation, including the Law on Cybersecurity 2018 and Decree  13/2023/ND-CP on Personal Data Protection, and reflects internationally recognised privacy  principles including those embodied in the General Data Protection Regulation (GDPR).  Customers are encouraged to read this Section carefully and to direct any questions to  privacy@Sanquo.vn.  5.1  Information Collection  Sanquo col

## Shipping

In [21]:
pages_and_texts = document_data[5]['doc']
split_sentence(pages_and_texts)
# chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 5/5 [00:00<00:00, 53.69it/s]


[{'page_number': 0,
  'page_char_count': 2204,
  'page_word_count': 321,
  'page_token_count': 551.0,
  'text': "Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 6: Shipping and Delivery  Sanquo is committed to delivering every order reliably, transparently, and on time. This  Section describes the shipping options available to customers, the estimated delivery  timelines by geographic zone, the process for tracking a shipment, the restrictions that apply  to certain addresses and product categories, and the procedures in place should a package  arrive damaged or fail to arrive at all. Customers are encouraged to review this Section  before placing an order to ensure that the delivery options available meet their requirements.  6.1  Shipping Options  Sanquo offers three shipping tiers, designed to accommodate a range of delivery priorities  and budgets. All options are fulfilled through Sanquo's contracted logistics network, which  i

## Technical issue

In [22]:
pages_and_texts = document_data[6]['doc']
split_sentence(pages_and_texts)
# chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 6/6 [00:00<00:00, 85.27it/s]


[{'page_number': 0,
  'page_char_count': 1583,
  'page_word_count': 225,
  'page_token_count': 395.75,
  'text': "Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 7: Technical and Device Issues  Sanquo is committed to delivering a stable, responsive, and technically reliable platform  experience. This Section sets out the system requirements for optimal Platform  performance, the procedures governing scheduled and emergency maintenance, practical  troubleshooting guidance for the most common technical issues encountered by customers,  and the scope of Sanquo's liability in relation to device-side and connectivity-related  disruptions.  7.1  System Requirements  To ensure the best possible experience on the Sanquo Platform — whether accessed via  web browser or the Sanquo mobile application — customers' devices should meet the  following technical specifications. Using unsupported browsers, operating systems, or  devices may result in

## Contact

In [23]:
pages_and_texts = document_data[7]['doc']
split_sentence(pages_and_texts)
# chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 4/4 [00:00<00:00, 119.07it/s]


[{'page_number': 0,
  'page_char_count': 1654,
  'page_word_count': 229,
  'page_token_count': 413.5,
  'text': 'Sanquo  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 8: Contact Information  Sanquo is here whenever customers need assistance. Whether the matter relates to an  order, a payment, an account, a return, or any other aspect of the Platform experience, our  support team is ready to help through a range of accessible channels. This Section sets out  how to reach us, when we are available, where we are located, and the response time  commitments we hold ourselves to.  8.1  Customer Support Channels  Sanquo provides customer support through the following official channels. Customers are  encouraged to select the channel best suited to the urgency and nature of their enquiry.  Channel  Contact Detail  Best Used For  Hotline  +84 1234 1423  Urgent matters requiring  real-time voice assistance  General Support Email  support@Sanquo.vn 

In [24]:
document_data

[{'name': 'PlatformIntroduction',
  'doc': [{'page_number': 0,
    'page_char_count': 1933,
    'page_word_count': 302,
    'page_token_count': 483.25,
    'text': 'SANQUO  Terms of Service & Business Policies  Effective Date: 30 April 2026    Version 1.0    Section 1: Introduction  1.1  Welcome to Sanquo  Welcome to Sanquo — Vietnam\'s dedicated fashion destination, built for customers who  believe that style is personal, and shopping should be effortless.  We created Sanquo with a single purpose: to provide a seamless, secure, and transparent  shopping experience that empowers you to discover and acquire the fashion products you  love with total confidence and ease. Whether you are exploring the latest trends, seeking a  wardrobe staple, or discovering an emerging Vietnamese designer, Sanquo is the place  where your style journey begins.  Your trust is the foundation upon which everything we do is built. We are committed to  earning it every day — through honest communication, a care

Embedding

In [25]:
from langchain_core.documents import Document

# --- Tunable Parameters ---
# You can experiment with these values
CHUNK_SIZE_IN_SENTENCES = 8  # How many sentences to include in one chunk
OVERLAP_IN_SENTENCES = 2     # How many sentences to overlap between chunks

# This will be our new, flat list for the vector store
all_chunks_for_vectorstore = []

# Your document_data list
for doc in document_data:
    doc_name = doc['name']
    
    # Iterate through each page in the document
    for page in doc['doc']:
        page_num = page['page_number']
        
        # --- This is the key change ---
        # We get the flat list of all sentences on the page
        sentences = page['sentences'] 
        
        if not sentences:
            continue

        # Calculate the step size (how many sentences to "slide" forward)
        # If chunk is 7 and overlap is 2, we "slide" 5 sentences
        step = CHUNK_SIZE_IN_SENTENCES - OVERLAP_IN_SENTENCES
        
        # --- Sliding Window Logic ---
        for i in range(0, len(sentences), step):
            
            # Get the chunk of sentences for this window
            chunk_sentences = sentences[i : i + CHUNK_SIZE_IN_SENTENCES]
            
            if not chunk_sentences:
                continue

            # Join the sentences back together to form the chunk's content
            chunk_text = " ".join(chunk_sentences)
            
            # Create the metadata for this specific chunk
            chunk_metadata = {
                "source_document": doc_name,
                "page": page_num,
                "chunk_start_sentence_index": i # Good for debugging
            }
            
            # Create the final Document object
            new_doc = Document(
                page_content=chunk_text,
                metadata=chunk_metadata
            )
            
            all_chunks_for_vectorstore.append(new_doc)

print(f"Created {len(all_chunks_for_vectorstore)} total OVERLAPPING chunks.")



Created 111 total OVERLAPPING chunks.


In [28]:
import os
from pathlib import Path  # Standard library in Python 3
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

# 1. Define base directory relative to this script file
# This points to the folder containing this script
current_dir = Path.cwd()



# 2. Define the persist directory dynamically
# This creates: .../backend_ecommerce/ai-module/chroma_newest
persist_directory = current_dir / "chroma_newest"

# Ensure the directory exists (good practice)
persist_directory.mkdir(parents=True, exist_ok=True)

print(f"Persisting data to: {persist_directory}")

# --- Configuration ---
if not os.environ.get("GOOGLE_API_KEY"):
    print("Warning: GOOGLE_API_KEY not found in environment variables.")

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

collection_name = "rag_document"

# Note: Convert Path object to string for libraries that expect strings
import time
from langchain_chroma import Chroma

# 1. Khởi tạo Vectorstore rỗng (chưa đưa data vào vội)
vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
    collection_name=collection_name
)

# 2. Định nghĩa số lượng chunk mỗi lần nhúng
# Set khoảng 50-80 để an toàn dưới mức 100 requests/phút
batch_size = 50 

# 3. Chạy vòng lặp để đưa data vào từ từ
print(f"Bắt đầu nhúng tổng cộng {len(all_chunks_for_vectorstore)} chunks...")

for i in range(0, len(all_chunks_for_vectorstore), batch_size):
    batch = all_chunks_for_vectorstore[i : i + batch_size]
    print(f"Đang xử lý batch từ {i} đến {i + len(batch)}...")
    
    # Đưa batch này vào Chroma
    vectorstore.add_documents(documents=batch)
    
    # Kiểm tra xem nếu chưa phải batch cuối cùng thì cho code nghỉ ngơi
    if i + batch_size < len(all_chunks_for_vectorstore):
        print("-> Đang đợi 40 giây để hồi năng lượng (tránh Rate Limit API)...")
        time.sleep(40) # Nghỉ 40s cho chắc ăn vì API bắt nghỉ ~31s

print("Hoàn tất nhúng và lưu trữ mà không bị API chửi!")

Persisting data to: e:\Study\Capstone project\backend_ecommerce\ai-module\main\chroma_newest
Bắt đầu nhúng tổng cộng 111 chunks...
Đang xử lý batch từ 0 đến 50...
-> Đang đợi 40 giây để hồi năng lượng (tránh Rate Limit API)...
Đang xử lý batch từ 50 đến 100...
-> Đang đợi 40 giây để hồi năng lượng (tránh Rate Limit API)...
Đang xử lý batch từ 100 đến 111...
Hoàn tất nhúng và lưu trữ mà không bị API chửi!


In [30]:
count = vectorstore._collection.count()
print(f"Loaded VectorStore with {count} documents.")

Loaded VectorStore with 111 documents.
